# 03 — Parameter Sweep: SNR vs α + Failure Mode Analysis

This notebook quantifies **how amplification quality degrades as α increases** and documents known failure modes.

**Temporal SNR** is used as the quality metric:
```
signal_power = mean temporal variance of bandpass-filtered luma in the signal ROI
noise_power  = mean temporal variance of bandpass-filtered luma in a static background ROI
SNR_dB       = 10 * log10(signal_power / noise_power)
```
A higher SNR means the amplified signal is stronger relative to the background noise.  As α increases beyond the optimal point, noise amplification dominates and SNR drops.

> **Run from the project root directory.**  Set all variables in the *Configuration* cell before running.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.io.video_io import load_video, save_video
from src.magnification.eulerian import run_eulerian
from src.magnification.phase_based import run_phase_based
from src.utils.metrics import compute_temporal_snr, snr_alpha_sweep
from src.visualization.render import render_side_by_side

print('Imports OK')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
VIDEO_PATH  = str(PROJECT_ROOT / 'videos' / 'wrist.avi')
SCALE       = 1.0

FREQ_LOW    = 0.5   # Hz
FREQ_HIGH   = 2.0   # Hz
LEVELS      = 4     # Laplacian pyramid levels (Eulerian)

# Alpha values to sweep
ALPHAS = [10, 25, 50, 75, 100, 150, 200]

# ROI bounding boxes: (x0, y0, x1, y1) in pixels
# Adjust these to match the scene:
#   SIGNAL_ROI — over the wrist / neck / pulse area
#   NOISE_ROI  — static background region with no expected motion
SIGNAL_ROI = (50, 50, 150, 150)    # (x0, y0, x1, y1)
NOISE_ROI  = (10, 10,  50,  50)

RESULTS_DIR = PROJECT_ROOT / 'results'
# ──────────────────────────────────────────────────────────────────────────

In [ ]:
frames, fps = load_video(VIDEO_PATH, scale=SCALE)
T, H, W, C = frames.shape
print(f'{T} frames  |  {W}×{H} px  |  {fps:.2f} fps')

# Confirm ROIs are within bounds
for name, roi in [('SIGNAL_ROI', SIGNAL_ROI), ('NOISE_ROI', NOISE_ROI)]:
    x0, y0, x1, y1 = roi
    assert 0 <= x0 < x1 <= W and 0 <= y0 < y1 <= H, \
        f'{name} {roi} is out of bounds for {W}×{H} frame — adjust the values above.'
print('ROIs OK')

## SNR vs α — Eulerian Pipeline

For each α value in `ALPHAS`, the Eulerian pipeline is run and the temporal SNR of the output is measured.  Expect SNR to peak near the optimal α and decline at higher values as noise dominates.

In [ ]:
snr_plot_path = str(RESULTS_DIR / 'plots' / 'snr_sweep.png')
(RESULTS_DIR / 'plots').mkdir(parents=True, exist_ok=True)

snr_results = snr_alpha_sweep(
    frames=frames,
    fps=fps,
    freq_low=FREQ_LOW,
    freq_high=FREQ_HIGH,
    alphas=ALPHAS,
    levels=LEVELS,
    signal_roi=SIGNAL_ROI,
    noise_roi=NOISE_ROI,
    output_path=snr_plot_path,
)

print('\nSNR results:')
for alpha, snr in sorted(snr_results.items()):
    print(f'  α={alpha:>5.0f}  →  SNR = {snr:+.2f} dB')

In [ ]:
# Inline plot
sorted_alpha = sorted(snr_results.keys())
snr_vals = [snr_results[a] for a in sorted_alpha]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sorted_alpha, snr_vals, marker='o', linewidth=2, color='steelblue')
ax.set_xlabel('Amplification factor α', fontsize=13)
ax.set_ylabel('Temporal SNR (dB)', fontsize=13)
ax.set_title(f'SNR vs α  |  [{FREQ_LOW}–{FREQ_HIGH}] Hz  |  Eulerian pipeline', fontsize=13)
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## Eulerian vs Phase-Based Comparison

Run both pipelines on the same clip with their recommended α values and compare the output visually.

In [ ]:
# Recommended α for this scene
EVM_ALPHA   = 50.0
PHASE_ALPHA = 25.0

evm_out = run_eulerian(
    frames=frames, fps=fps,
    freq_low=FREQ_LOW, freq_high=FREQ_HIGH,
    alpha=EVM_ALPHA, levels=LEVELS,
)

phase_out = run_phase_based(
    frames=frames, fps=fps,
    freq_low=FREQ_LOW, freq_high=FREQ_HIGH,
    alpha=PHASE_ALPHA,
    colorspace='luma3', pyramid_type='half_octave',
    sigma=2.0, batch_size=4,
)

In [ ]:
# Side-by-side comparison: original | EVM | phase-based
mid_frame = T // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [
    (frames,    f'Original'),
    (evm_out,   f'Eulerian (α={EVM_ALPHA})'),
    (phase_out, f'Phase-based (α={PHASE_ALPHA})'),
]
for ax, (vid, title) in zip(axes, pairs):
    ax.imshow(cv2.cvtColor((vid[mid_frame] * 255).astype('uint8'), cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')
plt.suptitle(f'Frame {mid_frame} comparison', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Save side-by-side comparison videos
(RESULTS_DIR / 'videos').mkdir(parents=True, exist_ok=True)

evm_vs_orig   = render_side_by_side(frames, evm_out)
phase_vs_orig = render_side_by_side(frames, phase_out)

save_video(evm_vs_orig,   str(RESULTS_DIR / 'videos' / 'comparison_evm.mp4'),   fps=fps)
save_video(phase_vs_orig, str(RESULTS_DIR / 'videos' / 'comparison_phase.mp4'), fps=fps)
print('Saved comparison videos.')

## Failure Mode Analysis

### Failure 1 — α Too High (Noise Amplification)

When α is far above the optimal value, **sensor noise is amplified** along with the signal.  The output develops a flickering, noisy appearance.  The Eulerian pipeline is more susceptible than phase-based because intensity noise scales with α, whereas phase noise is partially suppressed by amplitude weighting.

**Root cause:** The Laplacian pyramid bands contain both motion signal and sensor noise at every scale.  Multiplying by a large α amplifies both equally.

**Mitigation:** Keep α below the SNR-drop-off point identified in the sweep above.

In [ ]:
# Demo: Eulerian with very high α
HIGH_ALPHA = 200.0

evm_high = run_eulerian(
    frames=frames, fps=fps,
    freq_low=FREQ_LOW, freq_high=FREQ_HIGH,
    alpha=HIGH_ALPHA, levels=LEVELS,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (vid, title) in zip(axes, [
    (evm_out,  f'Eulerian α={EVM_ALPHA} (good)'),
    (evm_high, f'Eulerian α={HIGH_ALPHA} (noise-dominated)'),
]):
    ax.imshow(cv2.cvtColor((vid[mid_frame] * 255).astype('uint8'), cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Failure 2 — Wrong Frequency Band

If the bandpass window does not contain the signal frequency, the filtered signal is near zero and amplification produces nothing — or amplifies out-of-band noise instead.

**Root cause:** Mismatched frequency settings relative to the actual physiological rate (e.g., using 2–4 Hz for a resting pulse that sits at ~1.1 Hz).

**Mitigation:** Verify the target frequency with a reference measurement (heart rate monitor, breathing count) before selecting `FREQ_LOW` and `FREQ_HIGH`.

### Failure 3 — Camera Shake

Unintentional camera motion creates large, low-frequency phase changes that dominate the motion signal.  The amplified output shows global scene oscillation rather than localised physiological motion.

**Root cause:** The pyramid bandpass filter cannot distinguish *camera motion* from *scene motion* at the same frequency.

**Mitigation:** Pre-stabilize the video with `src/stabilization/raft_stabilize.py` before running the magnification pipeline:

```bash
python -m src.stabilization.raft_stabilize \
    --input  videos/shaky.mp4 \
    --output videos/stabilized.mp4 \
    --model  small
```

### Failure 4 — Pyramid Boundary Artefacts (Flickering Edges)

At pyramid level boundaries (near the edges of the Laplacian levels), the `pyrUp` reconstruction can introduce ringing artefacts that appear as flickering lines along high-contrast edges.

**Root cause:** Finite filter support in the Laplacian pyramid and imperfect reconstruction near image borders.

**Mitigation:** Increase `LEVELS` to spread the signal across more levels (so each level's contribution is smaller), or apply a gentle spatial Gaussian blur to the filtered levels before amplification.